In [2]:
using CSV
using DataFrames
using Dates
using JSON
using XLSX
using StatsBase
function time_to_minutes(time)
    return Dates.hour(time) * 60 + Dates.minute(time)
end

time_to_minutes (generic function with 1 method)

In [8]:
file = "INSTANCES/OAMRP-Data/20-aircraft/Flights_on_my_way.xlsx"
df_flights = DataFrame(XLSX.readtable(file, "Sheet1"))
df_flights.YEAR = year.(df_flights.Date)
df_flights.MONTH = month.(df_flights.Date)
df_flights.DAY = day.(df_flights.Date)
df_flights
select!(df_flights, Not(:Date))
rename!(df_flights, :Departure => :DEPARTURE_TIME)
rename!(df_flights, :Arrival => :ARRIVAL_TIME)
rename!(df_flights, :From => :ORIGIN_AIRPORT)
rename!(df_flights, :To => :DESTINATION_AIRPORT)

Row,ORIGIN_AIRPORT,DEPARTURE_TIME,DESTINATION_AIRPORT,ARRIVAL_TIME,Day Change,YEAR,MONTH,DAY
,Any,Any,Any,Any,Any,Int64,Int64,Int64
1,DIY,06:30:00,ESB,07:55:00,0,2012,3,19
2,ADA,07:00:00,SAW,08:30:00,0,2012,3,19
3,ADB,07:00:00,ESB,08:15:00,0,2012,3,19
4,ASR,07:00:00,SAW,08:20:00,0,2012,3,19
5,AYT,07:00:00,ESB,08:05:00,0,2012,3,19
6,ECN,07:00:00,ESB,08:05:00,0,2012,3,19
7,EZS,07:00:00,ESB,08:20:00,0,2012,3,19
8,GZT,07:00:00,ESB,08:10:00,0,2012,3,19
9,MLX,07:00:00,ESB,08:15:00,0,2012,3,19


In [9]:
for row in eachrow(df_flights)
    dt = time_to_minutes(row.DEPARTURE_TIME) + 1440*(row.DAY-1)
    at = time_to_minutes(row.ARRIVAL_TIME) + 1440*(row.DAY-1)
    if at < dt
        at += 1440
    end 
    row.DEPARTURE_TIME = dt
    row.ARRIVAL_TIME = at 
end 
df_flights.AIR_TIME = df_flights.ARRIVAL_TIME .- df_flights.DEPARTURE_TIME
df_flights
#XLSX.writetable(airline*"_2024-0"*string(m)*"_real_flights.xlsx", df_flights, overwrite = true)

Row,ORIGIN_AIRPORT,DEPARTURE_TIME,DESTINATION_AIRPORT,ARRIVAL_TIME,Day Change,YEAR,MONTH,DAY,AIR_TIME
,Any,Any,Any,Any,Any,Int64,Int64,Int64,Int64
1,DIY,26310,ESB,26395,0,2012,3,19,85
2,ADA,26340,SAW,26430,0,2012,3,19,90
3,ADB,26340,ESB,26415,0,2012,3,19,75
4,ASR,26340,SAW,26420,0,2012,3,19,80
5,AYT,26340,ESB,26405,0,2012,3,19,65
6,ECN,26340,ESB,26405,0,2012,3,19,65
7,EZS,26340,ESB,26420,0,2012,3,19,80
8,GZT,26340,ESB,26410,0,2012,3,19,70
9,MLX,26340,ESB,26415,0,2012,3,19,75


In [10]:
df_flights = df_flights[:, ["YEAR", "MONTH", "DAY", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DEPARTURE_TIME", "AIR_TIME", "ARRIVAL_TIME"]]

Row,YEAR,MONTH,DAY,ORIGIN_AIRPORT,DESTINATION_AIRPORT,DEPARTURE_TIME,AIR_TIME,ARRIVAL_TIME
,Int64,Int64,Int64,Any,Any,Any,Int64,Any
1,2012,3,19,DIY,ESB,26310,85,26395
2,2012,3,19,ADA,SAW,26340,90,26430
3,2012,3,19,ADB,ESB,26340,75,26415
4,2012,3,19,ASR,SAW,26340,80,26420
5,2012,3,19,AYT,ESB,26340,65,26405
6,2012,3,19,ECN,ESB,26340,65,26405
7,2012,3,19,EZS,ESB,26340,80,26420
8,2012,3,19,GZT,ESB,26340,70,26410
9,2012,3,19,MLX,ESB,26340,75,26415


In [12]:
XLSX.writetable("667FL_20A.xlsx", df_flights, overwrite = true)
